# Lec 01 — TensorFlow 기본 (TF 2.x)

원본 강의는 TF 1.x (`Session`, `placeholder`) 기반이지만, 본 실습은 **TF 2.x Eager execution** 으로 진행합니다.

다루는 내용:
1. 환경 확인 (버전, 디바이스)
2. `tf.constant` — 상수 텐서
3. 기본 연산 / 브로드캐스팅
4. `tf.Variable` — 학습 가능한 변수
5. `@tf.function` — 그래프 컴파일
6. (참고) TF 1.x `Session` 스타일과의 비교

## 0. 환경 확인

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import tensorflow as tf
import numpy as np

print("TensorFlow:", tf.__version__)
print("NumPy     :", np.__version__)
print("Eager     :", tf.executing_eagerly())
print("GPUs      :", tf.config.list_physical_devices("GPU"))

## 1. `tf.constant` — Hello TensorFlow

TF 1.x 강의의 첫 예제 `hello = tf.constant("Hello, TensorFlow!")` 를 2.x 스타일로.

In [ ]:
hello = tf.constant("Hello, TensorFlow!")
print(hello)
print("value:", hello.numpy().decode())

In [ ]:
scalar = tf.constant(3.0)
vector = tf.constant([1.0, 2.0, 3.0])
matrix = tf.constant([[1, 2], [3, 4]])

for name, t in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(f"{name:6s} shape={t.shape} dtype={t.dtype}")
    print(t.numpy(), "\n")

## 2. 기본 연산 (Computational Graph)

TF 1.x: 그래프를 만든 뒤 `Session.run()` 으로 실행 → TF 2.x: **즉시 실행**(eager). 그래프는 `@tf.function` 으로 명시적으로 만들 수 있음.

In [ ]:
a = tf.constant(3.0)
b = tf.constant(4.0)

print("a + b =", (a + b).numpy())
print("a * b =", (a * b).numpy())
print("a ** 2 =", tf.pow(a, 2).numpy())

In [ ]:
x = tf.constant([1.0, 2.0, 3.0])
y = tf.constant([10.0, 20.0, 30.0])

print("x + y      :", (x + y).numpy())
print("dot(x, y)  :", tf.tensordot(x, y, axes=1).numpy())
print("x * 2 (br) :", (x * 2).numpy())

## 3. `tf.Variable` — 학습 가능한 파라미터

이후 강의(선형회귀)에서 `W`, `b` 를 표현하는 데 쓰임.

In [ ]:
W = tf.Variable(0.5, name="W")
b = tf.Variable(0.1, name="b")

print("before:", W.numpy(), b.numpy())

W.assign(1.0)
b.assign_add(0.5)

print("after :", W.numpy(), b.numpy())

## 4. `@tf.function` — 함수를 그래프로

Python 함수를 데코레이트하면 첫 호출 시 그래프로 컴파일되고 이후 호출이 빨라짐. 강의에서 `Session.run(op, feed_dict={...})` 에 해당하는 추상화.

In [ ]:
@tf.function
def linear(x, W, b):
    return W * x + b

x = tf.constant([1.0, 2.0, 3.0])
print("linear(x):", linear(x, W, b).numpy())

## 5. (참고) TF 1.x 스타일과 비교

원본 강의 코드:

```python
# TF 1.x — 실행 안 됨
hello = tf.constant("Hello, TensorFlow!")
sess = tf.Session()
print(sess.run(hello))
```

TF 2.x 동치:

```python
hello = tf.constant("Hello, TensorFlow!")
print(hello.numpy().decode())
```

TF 1.x → 2.x 핵심 변화:

| TF 1.x | TF 2.x |
| --- | --- |
| `tf.placeholder` + `feed_dict` | 함수 인자 |
| `tf.Session().run(op)` | 텐서를 그냥 호출 (eager) |
| `tf.global_variables_initializer()` | `tf.Variable` 정의 시 자동 초기화 |
| Static graph | `@tf.function` 으로 옵트인 |

## 마무리 — 체크리스트

- [ ] `tf.constant` 로 스칼라/벡터/행렬 만들기
- [ ] `+`, `*`, `tf.tensordot` 등 기본 연산
- [ ] `tf.Variable` 의 `assign` / `assign_add`
- [ ] `@tf.function` 으로 그래프 컴파일
- [ ] Session/placeholder 가 더 이상 필요 없는 이유 이해